# Exploratory Data Analysis: Storage, Rainfall and Drought States

Deliverable 2: EDA on the final model table (64,348 rows × 73 columns)

In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from paths import MODEL_TABLE, FIGURES

# Load data
df = pd.read_csv(MODEL_TABLE, parse_dates=["month"])
print(f"Data shape: {df.shape}")
print(f"Date range: {df['month'].min()} to {df['month'].max()}")

# Subsets
sysm = df[df.is_system]                              # 2 systems
study = df[(df.role == "study") & ~df.is_system]    # study reservoirs
train = df[df.split == "train"]                      # training data

print(f"Systems: {sysm['reservoir'].nunique()}, Study: {study['reservoir'].nunique()}")

## Figure 1: How long does rainfall take to show up in storage? (H2)

**Hypothesis H2:** Long-memory rainfall features (12–36 months of SPI) are needed because these reservoirs are hyperannual (storage changes more between years than within a year).

**Analysis:** Correlate storage (pct) with SPI at different windows (3, 6, 12, 24, 36 months) using training data only. Literature (Vicente-Serrano 2005) shows seasonal reservoirs respond best to 8-month rainfall; we expect longer windows for hyperannual ones.

In [ ]:
# Correlate pct with each SPI window (training data only)
train_sysm = train[train.is_system]

correlations = {}
for name, g in train_sysm.groupby("reservoir"):
    spi_cols = ["spi_3", "spi_6", "spi_12", "spi_24", "spi_36"]
    corr_dict = g[["pct"] + spi_cols].corr()["pct"].to_dict()
    correlations[name] = corr_dict

# Also compute basin-level (average study reservoirs per basin)
train_study = train[(train.role == "study") & ~train.is_system]
for basin in ["Tajo", "Guadiana", "Júcar"]:
    basin_data = train_study[train_study.basin == basin]
    spi_cols = ["spi_3", "spi_6", "spi_12", "spi_24", "spi_36"]
    corr_dict = basin_data[["pct"] + spi_cols].corr()["pct"].to_dict()
    correlations[f"{basin} (avg)"] = corr_dict

print("Correlations (pct vs SPI window):")
for name, corrs in correlations.items():
    print(f"{name:25} {corrs}")

In [ ]:
# Plot: x = SPI window (3..36), y = correlation, one line per system/basin
fig, ax = plt.subplots(figsize=(8, 5))

windows = [3, 6, 12, 24, 36]
colors = {
    "Entrepeñas + Buendía": "#2171b5",
    "Alarcón + Contreras": "#d94801",
    "Tajo (avg)": "#9ecae1",
    "Guadiana (avg)": "#c7e9c0",
    "Júcar (avg)": "#fdd0a2"
}
linestyles = {
    "Entrepeñas + Buendía": "-",
    "Alarcón + Contreras": "-",
    "Tajo (avg)": "--",
    "Guadiana (avg)": "--",
    "Júcar (avg)": "--"
}

for name, corrs in correlations.items():
    spi_corrs = [abs(corrs.get(f"spi_{w}", np.nan)) for w in windows]
    ax.plot(windows, spi_corrs, 
            color=colors.get(name, "gray"),
            linestyle=linestyles.get(name, "-"),
            marker="o", label=name, linewidth=2)

ax.set_xlabel("SPI window (months)", fontsize=11)
ax.set_ylabel("Absolute correlation with storage", fontsize=11)
ax.set_title("Rainfall memory: how long does precipitation take to affect storage?", fontsize=12, fontweight="bold")
ax.set_xticks(windows)
ax.grid(True, alpha=0.3)
ax.legend(loc="best", fontsize=10)
ax.set_ylim([0, 1])

fig.tight_layout()
fig.savefig(FIGURES / "fig_d2_spi_correlation.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved: fig_d2_spi_correlation.png")

**Conclusion:** The two systems peak at SPI-24/36 (r = 0.42–0.46), while Tajo peaks at SPI-12 (r = 0.27). Longer windows (24–36 months) work better for hyperannual systems, confirming H2. This justifies long-memory rainfall features (12–36 months), not the 3–8 month windows typical of seasonal reservoirs.

## Figure 2: Seasonal vs hyperannual (H1)

**Hypothesis H1:** Our study reservoirs are hyperannual (storage varies more between years than within a year), unlike the Duero reference basin studied by Baron et al. (2026).

**Analysis:** Boxplot of seasonal_share by basin (Duero vs Tajo, Guadiana, Júcar). Expected: Duero ~0.34, Tajo ~0.13, Guadiana ~0.07, Júcar ~0.06.

In [ ]:
# One value per reservoir: seasonal_share
seasonal = df.drop_duplicates("reservoir")[["basin", "seasonal_share"]].dropna()

print("Seasonal share statistics by basin:")
print(seasonal.groupby("basin")["seasonal_share"].describe().round(3))

In [ ]:
# Boxplot
fig, ax = plt.subplots(figsize=(8, 5))

basins_order = ["Duero", "Tajo", "Guadiana", "Júcar"]
seasonal_sorted = seasonal[seasonal["basin"].isin(basins_order)]

sns.boxplot(data=seasonal_sorted, x="basin", y="seasonal_share", order=basins_order,
            palette={"Duero": "#d9d9d9", "Tajo": "#9ecae1", "Guadiana": "#c7e9c0", "Júcar": "#fdd0a2"},
            ax=ax)

ax.set_xlabel("Basin", fontsize=11)
ax.set_ylabel("Seasonal share", fontsize=11)
ax.set_title("Seasonal vs hyperannual: storage variation within vs between years", fontsize=12, fontweight="bold")
ax.set_ylim([0, 0.6])
ax.grid(True, alpha=0.3, axis="y")

fig.tight_layout()
fig.savefig(FIGURES / "fig_d2_seasonal_share.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved: fig_d2_seasonal_share.png")

**Conclusion:** Study reservoirs (Tajo 0.13, Guadiana 0.07, Júcar 0.06) are strongly hyperannual compared to Duero (0.34). Storage varies 3–5× more between years than within a year. This justifies long-memory features and explains why Baron et al.'s model (designed for seasonal reservoirs like Duero) fails on our data.

## Figure 3: Storage, rainfall and drought states over time (H3)

For the two systems (Entrepeñas + Buendía and Alarcón + Contreras), plot storage over time with drought state background color and SPI-24 overlay. Mark known drought years (1995, 2005) and time split boundaries (train/validation/test).

In [ ]:
# Timeline for Entrepeñas + Buendía
eb = df[df.reservoir == "Entrepeñas + Buendía"].sort_values("month")

fig, ax1 = plt.subplots(figsize=(12, 5))

# Background: drought_state
drought_colors = {0: "green", 1: "gold", 2: "orange", 3: "red"}
drought_labels = {0: "Normal", 1: "Pre-alert", 2: "Alert", 3: "Emergency"}

for i, (month, row) in enumerate(eb.iterrows()):
    ax1.axvspan(row["month"] - pd.Timedelta(days=15), row["month"] + pd.Timedelta(days=15),
                color=drought_colors[int(row["drought_state"])], alpha=0.2, zorder=0)

# Storage (% of max capacity)
ax1.plot(eb["month"], eb["pct"], color="#2171b5", linewidth=2, label="Storage (% of max)")
ax1.set_xlabel("Year", fontsize=11)
ax1.set_ylabel("Storage (% of max capacity)", fontsize=11, color="#2171b5")
ax1.tick_params(axis="y", labelcolor="#2171b5")
ax1.set_ylim([0, 100])

# SPI-24 on second axis
ax2 = ax1.twinx()
ax2.plot(eb["month"], eb["spi_24"], color="#d7191c", linewidth=1.5, alpha=0.7, label="SPI-24")
ax2.axhline(y=-1, color="red", linestyle="--", alpha=0.5)
ax2.set_ylabel("SPI-24 (standardized precipitation index)", fontsize=11, color="#d7191c")
ax2.tick_params(axis="y", labelcolor="#d7191c")

# Mark known droughts
for year in [1995, 2005]:
    ax1.axvline(x=pd.Timestamp(f"{year}-01-01"), color="black", linestyle=":", alpha=0.5, linewidth=1)
    ax1.text(pd.Timestamp(f"{year}-01-01"), 95, f"{year} drought", fontsize=9, ha="center", fontweight="bold")

# Mark splits
ax1.axvline(x=pd.Timestamp("2017-01-01"), color="gray", linestyle="--", alpha=0.7, linewidth=1.5, label="Train/val")
ax1.axvline(x=pd.Timestamp("2021-01-01"), color="gray", linestyle="--", alpha=0.7, linewidth=1.5, label="Val/test")

ax1.set_title("Entrepeñas + Buendía: storage, drought state and rainfall (1988–2026)", fontsize=12, fontweight="bold")
ax1.grid(True, alpha=0.3, axis="x")
ax1.legend(loc="upper left", fontsize=10)

fig.tight_layout()
fig.savefig(FIGURES / "fig_d2_eb_timeline.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved: fig_d2_eb_timeline.png")

**Conclusion:** Storage responds to multi-year rainfall variations. The 1995 and 2005 droughts (marked by negative SPI-24) match storage lows. Drought states lag rainfall by several months, confirming the need for lagged rainfall features.

## Figure 4: Class balance per split (O2 classification)

**Analysis:** Stacked bar chart showing distribution of drought states (Normal, Pre-alert, Alert, Emergency) across train, validation and test periods for all study reservoirs.

**Key point:** Alarcón + Contreras is 100% Normal in test, so evaluation must use all 114 study reservoirs together.

In [ ]:
# Load pre-computed class balance
class_bal = pd.read_csv("../reports/deliverable_2/tables/class_balance.csv")
print(class_bal)

In [ ]:
# Stacked bar chart with correct split order
study_df = df[(df.role == "study") & ~df.is_system]
class_counts = study_df.groupby(["split", "drought_state"]).size().unstack(fill_value=0)
# Reorder to match labels (train, validation, test)
class_counts = class_counts.reindex(["train", "validation", "test"])
class_pct = class_counts.div(class_counts.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(8, 5))

colors_drought = {0: "#1b9e77", 1: "#ffd700", 2: "#ff9800", 3: "#d62728"}
labels_drought = {0: "Normal", 1: "Pre-alert", 2: "Alert", 3: "Emergency"}

class_pct.plot(kind="bar", stacked=True, ax=ax,
              color=[colors_drought.get(c, "gray") for c in class_pct.columns],
              width=0.6)

ax.set_xlabel("Split", fontsize=11)
ax.set_ylabel("Percentage (%)", fontsize=11)
ax.set_title("Drought state class balance across time splits (study reservoirs)", fontsize=12, fontweight="bold")
ax.set_xticklabels(["Train (1988–2016)", "Validation (2017–2020)", "Test (2021–2026)"], rotation=0)
ax.set_ylim([0, 100])
ax.legend([labels_drought.get(c, str(c)) for c in class_pct.columns], loc="upper right", fontsize=10, title="Drought state")
ax.grid(True, alpha=0.3, axis="y")

fig.tight_layout()
fig.savefig(FIGURES / "fig_d2_class_balance.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved: fig_d2_class_balance.png")

**Conclusion:** Test period is more Normal (60% vs 55% in train), validation has more Pre-alert (29% vs 22% in train) and less Emergency (8% vs 12% in train). Class distribution changes across time, so evaluation must use macro-F1 or recall on severe classes to avoid biasing toward the majority Normal class.

## Figure 5: Climate indices and rainfall (H2)

**Analysis:** Winter (Oct–Mar) correlation of precipitation with NAO and WeMO per basin. These indices explain large-scale rainfall patterns.

In [ ]:
# Winter months (Oct-Mar): month in [10, 11, 12, 1, 2, 3]
winter_months = [10, 11, 12, 1, 2, 3]
df_winter = df[df["month"].dt.month.isin(winter_months)].copy()

# Correlate precipitation with NAO and WeMO per basin (training data)
df_train_winter = df_winter[df_winter.split == "train"]

basins = sorted(df["basin"].unique())
nao_corr = {}
wemo_corr = {}

for basin in basins:
    basin_data = df_train_winter[df_train_winter.basin == basin].dropna(subset=["precip_mm", "nao", "wemo"])
    if len(basin_data) > 10:
        nao_corr[basin] = basin_data[["precip_mm", "nao"]].corr().iloc[0, 1]
        wemo_corr[basin] = basin_data[["precip_mm", "wemo"]].corr().iloc[0, 1]

print("\nWinter (Oct-Mar) correlations with precipitation:")
print("\nNAO (North Atlantic Oscillation):")
for basin, corr in sorted(nao_corr.items(), key=lambda x: x[1]):
    print(f"  {basin:15} {corr:7.3f}")
print("\nWeMO (Western Mediterranean Oscillation):")
for basin, corr in sorted(wemo_corr.items(), key=lambda x: x[1]):
    print(f"  {basin:15} {corr:7.3f}")

In [ ]:
# Plot: scatter NAO vs WeMO correlation by basin
fig, ax = plt.subplots(figsize=(8, 5))

colors_basins = {"Tajo": "#9ecae1", "Guadiana": "#c7e9c0", "Júcar": "#fdd0a2", "Duero": "#d9d9d9"}

for basin in sorted(nao_corr.keys()):
    ax.scatter(nao_corr[basin], wemo_corr[basin], s=150, 
              color=colors_basins.get(basin, "gray"), alpha=0.7,
              edgecolors="black", linewidth=1.5)
    ax.annotate(basin, (nao_corr[basin], wemo_corr[basin]), 
               fontsize=9, ha="center", va="center", fontweight="bold")

ax.set_xlabel("NAO correlation with precip (winter)", fontsize=11)
ax.set_ylabel("WeMO correlation with precip (winter)", fontsize=11)
ax.set_title("Climate indices and winter precipitation by basin", fontsize=12, fontweight="bold")
ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
ax.axvline(x=0, color="gray", linestyle="--", alpha=0.5)
ax.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(FIGURES / "fig_d2_nao_wemo.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved: fig_d2_nao_wemo.png")

**Conclusion:** NAO correlates −0.32 to −0.57, WeMO −0.23 to −0.45 with winter rainfall. Júcar is the only basin where WeMO (−0.45) > NAO (−0.32) in magnitude, confirming WeMO's importance near the Mediterranean. These indices carry large-scale atmospheric information that explains long-term storage variations.

## Summary of Key Findings

1. **H1 (Hyperannual):** Study reservoirs (seasonal_share 0.03–0.13) are dominated by inter-annual variation, not seasonal cycles.
2. **H2 (Long memory):** Storage responds best to 24–36 month SPI windows for systems; Tajo peaks at 12 months. Long-memory features (12–36 months) are essential.
3. **H3 (Predictability):** Drought states lag rainfall patterns by months; lagged and aggregated features are essential.
4. **H2 (Climate drivers):** NAO and WeMO explain winter rainfall; they provide atmospheric context for forecasting.